# Projeto 3 — execução dos seis treinos no Google Colab

**Autores:** Paulo Henrico Rabelo (2312652) e Joey Alan (2320416)

Em *Ambiente de execução → Alterar o tipo de ambiente de execução*, escolha **GPU (T4)**. O Colab apaga os arquivos ao desconectar,
por isso a última célula copia `modelos/` e `resultados/` para o Google Drive. Todos os seis treinos devem rodar **no mesmo
hardware**; se a sessão cair, o script pula os modelos já concluídos (copie-os de volta do Drive antes).

In [ ]:
!nvidia-smi

Repositório privado: crie um *token* do GitHub com acesso de leitura e use `https://<token>@github.com/...`.

In [ ]:
!git clone https://github.com/JoeyAlanS/redes-reurais-fotos-satelite.git
%cd redes-reurais-fotos-satelite

In [ ]:
!mkdir -p dados/raw
!curl -L --fail -s -o dados/raw/NWPU-RESISC45.zip 'https://huggingface.co/datasets/isaaccorley/resisc45/resolve/main/NWPU-RESISC45.zip'
!echo 'beeecd0b63656290ae6d65cf7763185b0c1c4c54a753ef8088d6fba3faaf1f53  dados/raw/NWPU-RESISC45.zip' | sha256sum -c -
!unzip -q dados/raw/NWPU-RESISC45.zip 'NWPU-RESISC45/airport/*' 'NWPU-RESISC45/beach/*' 'NWPU-RESISC45/circular_farmland/*' 'NWPU-RESISC45/dense_residential/*' 'NWPU-RESISC45/desert/*' 'NWPU-RESISC45/forest/*' 'NWPU-RESISC45/harbor/*' 'NWPU-RESISC45/mountain/*' 'NWPU-RESISC45/river/*' 'NWPU-RESISC45/storage_tank/*' -d dados/raw

In [ ]:
!pip install -q "tensorflow[and-cuda]==2.21.0" "scikit-learn>=1.5,<2" "pandas>=2.2" "matplotlib>=3.9" "python-dotenv>=1,<2"
!cp -n .env.example .env
!python -c "import tensorflow as tf; print(tf.__version__, tf.config.list_physical_devices('GPU'))"

Opcional: recuperar do Drive modelos já treinados numa sessão anterior.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DESTINO = '/content/drive/MyDrive/projeto3'
!mkdir -p {DESTINO}
!test -d {DESTINO}/modelos && cp -rn {DESTINO}/modelos/. modelos/ && cp -rn {DESTINO}/resultados/. resultados/ || true

Prepara as partições (resultado idêntico ao versionado), treina os seis modelos e avalia a **validação**.

In [ ]:
!PY=python bash scripts/executar_tudo.sh

In [ ]:
!rsync -a modelos resultados dados/splits {DESTINO}/

**Só depois de encerrar todas as escolhas**: previsão única no teste e avaliação final.

In [ ]:
!python scripts/prever.py --teste
!python scripts/avaliar.py
!rsync -a modelos resultados {DESTINO}/